# 립리딩 전처리 · 학습 파이프라인 (Colab)

원본 영상을 Google Drive에 두고, 코랩 GPU로 전처리와 학습을 수행한다.

**실행 전 준비**
1. 런타임 → 런타임 유형 변경 → 하드웨어 가속기 **GPU** 선택
2. Drive에 영상 폴더 생성 후 녹화본 업로드
3. 파일명 규칙: `{화자}_{문구}_{번호}.mp4` — 예) `s01_물주세요_01.mp4`

화자가 **2명 이상**이어야 학습이 진행된다. 화자 단위로 학습·검증을 나누기 때문이다.

## 1. 환경 확인

In [57]:
import torch

print(f"torch {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("경고: 런타임 유형을 GPU로 변경하세요.")

torch 2.11.0+cu128
CUDA 사용 가능: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Drive 마운트

In [58]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. 경로 설정

In [59]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_RAW = DRIVE_ROOT / "raw"
DRIVE_PROCESSED = DRIVE_ROOT / "processed"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"

for folder in (DRIVE_RAW, DRIVE_PROCESSED, DRIVE_CHECKPOINTS):
    folder.mkdir(parents=True, exist_ok=True)

videos = sorted(
    p.name for p in DRIVE_RAW.glob("*") if p.suffix.lower() in (".mp4", ".avi", ".mov")
)
print(f"영상 {len(videos)}개")
for name in videos[:10]:
    print(f"  {name}")

영상 1238개
  s01_가래가있어요_01.mp4
  s01_가래가있어요_02.mp4
  s01_가래가있어요_03.mp4
  s01_가래가있어요_04.mp4
  s01_가래가있어요_05.mp4
  s01_가래가있어요_06.mp4
  s01_가래가있어요_07.mp4
  s01_가래가있어요_08.mp4
  s01_가래가있어요_09.mp4
  s01_가래가있어요_10.mp4


## 4. 저장소 clone

In [60]:
import os

REPO_URL = "https://github.com/HumanRhoid/hanium-lipreading.git"
BRANCH = "develop"
REPO_DIR = Path("/content/hanium-lipreading")

# clone이 중간에 실패하면 빈 폴더만 남아 다음 실행에서 git 명령이 어긋난다.
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !rm -rf {REPO_DIR}
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"작업 경로: {Path.cwd()}")

Already on 'develop'
Your branch is up to date with 'origin/develop'.
Already up to date.
작업 경로: /content/hanium-lipreading


## 5. 의존성 설치

In [61]:
!pip install --quiet mediapipe opencv-python wandb

import torch

print(f"설치 후 CUDA 사용 가능: {torch.cuda.is_available()}")

설치 후 CUDA 사용 가능: True


## 6. 얼굴 랜드마크 모델 내려받기

In [62]:
LANDMARKER_URL = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task"
landmarker_path = REPO_DIR / "models" / "face_landmarker.task"
landmarker_path.parent.mkdir(parents=True, exist_ok=True)

if not landmarker_path.exists():
    !wget -q -O {landmarker_path} {LANDMARKER_URL}

print(f"{landmarker_path.name}: {landmarker_path.stat().st_size / 1e6:.1f} MB")

face_landmarker.task: 3.8 MB


## 7. 전처리 — 영상을 .npy로 변환

### (192x96 닮음 변환 정렬)

`e316d85`부터의 규격이다. 위 60프레임 셀과 달리 `vid2npy.process_video`를
갈아끼우지 않는다. `d5f47f2`에서 `frames`가 인자로 나왔으므로 몽키패치가
필요 없다.

In [ ]:
# 크롭 규격을 바꿨으면 반드시 새 폴더에 굽는다. run_batch는 이미 있는 파일을
# 건너뛰므로 같은 폴더에 다시 돌리면 옛 규격이 그대로 남고, 개수도 그대로라
# 다음 셀의 검사도 통과해 버린다.
PROCESSED_ALIGN = DRIVE_ROOT / "processed_align"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT, TARGET_HEIGHT, TARGET_WIDTH

print(f"규격 {TARGET_WIDTH}x{TARGET_HEIGHT} · {FIXED_FRAME_COUNT}프레임 -> {PROCESSED_ALIGN}")
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_ALIGN,
                  frames=FIXED_FRAME_COUNT)

## 8-2. 셋업을 한 셀로

Drive 마운트부터 매니페스트·로컬 복사까지 한 번에 돌린다. 런타임을 새로 켰을 때 2~5절과 8절 대신 이것만 실행하면 된다.

**두 셀 모두 npy가 1234개인지 검사한다.** 화자를 교체하면 개수가 달라지므로 그 줄을 고치거나 지워야 한다.

앞 셀은 `git clone` 줄이 깨져 있다. 뒤 셀이 고친 것이다.


In [73]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time, subprocess
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
# 크롭 규격마다 폴더가 다르다. 섞으면 백본이 첫 배치에서 죽는다.
#   processed_f60     112x80 · 축 정렬 상자        (옛 규격)
#   processed_align   192x96 · 닮음 변환 정렬      (2026-08-24~, 현재)
PROCESSED = DRIVE_ROOT / "processed_align"
PROCESSED_F60 = PROCESSED   # 아래 절들이 쓰는 옛 이름 호환

n_drive = len(list(PROCESSED.glob("*.npy")))
print("Drive", PROCESSED.name + ":", n_drive, "개")
assert n_drive == 1238, f"{PROCESSED.name}이 1238개가 아니다"

os.chdir("/content")
for junk in Path("/content").glob("*REPO_DIR*"):
    shutil.rmtree(junk, ignore_errors=True)
    print("잘못 만들어진 폴더 삭제:", junk.name)

REPO = "/content/hanium-lipreading"
REPO_DIR = Path(REPO)
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "develop"], check=True)
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "-b", "develop",
                    "https://github.com/HumanRhoid/hanium-lipreading.git", REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "wandb"], check=True)

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"

# 개수만 세면 같은 폴더에 다른 규격으로 다시 구웠을 때를 못 잡는다.
# 파일을 한 장 열어 백본이 요구하는 크기와 맞는지 본다.
import numpy as np
from src.ml.models.backbone import LipReadingBackbone
from src.ml.preprocess.normalize import FIXED_FRAME_COUNT

shape = np.load(next(LOCAL_F60.glob("*.npy"))).shape
want = (FIXED_FRAME_COUNT, LipReadingBackbone.input_height,
        LipReadingBackbone.input_width, 3)
assert shape == want, f"규격 불일치: npy {shape} · 백본 기대 {want}"
print("규격 확인", shape)
# end$0

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1238 개
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1238개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1238 개 · 0 초 · 0바이트 0 개


## 9. 학습

`train()`은 **딕셔너리**를 반환한다. 정확도만 쓸 때는 `["best"]`를 꺼낸다.

```
best · best_smoothed · best_epoch · peak · peak_epoch · saturation_epoch
errors · val_size · last · trainable_params · config
```

발표·서류 수치는 `["best"]`가 아니라 `["last"]`다. `best`는 검증 화자를 보고 고른
값이라 낙관 편향이 있다(관측 +0.037).

### (60 프레임)

In [76]:
# ═══ 60프레임 8화자 교차검증 · seed 42 ═══
from src.ml.training.train import train

SEEDS = [42, 1, 7]
speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS} · 60프레임\n")

results60cv = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'='*16} {speaker} · seed {seed} · 60프레임 {'='*16}")
        results60cv[(speaker, seed)] = train(
            manifest_path=manifest_f60,          # ← 60프레임
            data_root=TRAIN_ROOT_F60,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv60_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv60_{speaker}_seed{seed}",
        )["best"]

BASE30 = {"s01": 0.656, "s03": 0.446, "s04": 0.780, "s05": 0.307,
          "s06": 0.732, "s07": 0.351, "s08": 0.556, "s09": 0.393}

print(f"\n{'='*56}")
print(f"{'화자':<6}{'30프레임':>10}{'60프레임':>10}{'차이':>10}")
print("-" * 40)
per = {}
for sp in speakers:
    v = sum(results60cv[(sp, s)] for s in SEEDS) / len(SEEDS)
    per[sp] = v
    print(f"{sp:<6}{BASE30[sp]:>10.3f}{v:>10.3f}{v - BASE30[sp]:>+10.3f}")

new, old = sum(per.values()) / len(per), sum(BASE30.values()) / len(BASE30)
print("-" * 40)
print(f"{'평균':<6}{old:>10.3f}{new:>10.3f}{new - old:>+10.3f}")
print(f"\n화자 간 편차   30프레임 {max(BASE30.values())-min(BASE30.values()):.3f}"
      f" · 60프레임 {max(per.values())-min(per.values()):.3f}")

화자 ['s01', 's03', 's04', 's05', 's06', 's07', 's08', 's09'] · 시드 [42, 1, 7] · 60프레임


================ s01 · seed 42 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1081개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7094 acc 0.081 | val loss 2.7058 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.6183 acc 0.144 | val loss 2.6992 acc 0.064 avg 0.067 | lr 2.00e-04
[  3/80] train loss 2.5033 acc 0.250 | val loss 2.6981 acc 0.064 avg 0.066 | lr 1.99e-04
[  4/80] train loss 2.3507 acc 0.406 | val loss 2.7065 acc 0.064 avg 0.064 | lr 1.99e-04
[  5/80] train loss 2.1987 acc 0.607 | val loss 2.7329 acc 0.070 avg 0.066 | lr 1.98e-04
[  6/80] train loss 2.0314 acc 0.769 | val loss 2.7918 acc 0.070 avg 0.068 | lr 1.97e-04
[  7/80] train loss 1.9247 acc 0.867 | val loss 2.8748 acc 0.070 avg 0.070 | lr 1.96e-04
[  8/80] train loss 1.8699 acc 0.900 | val loss 2.9670 acc 0.070 avg 0.070 | lr 1.95e-04
[  9/80] train loss 1.8198 acc 0.934 | val loss

lr,███████▇▇▇▆▆▆▅▅▅▅▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
train/acc,▁▃▆▇████████████████████████████████████
train/loss,█▆▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▂▂▃▅▆███████▇▇▇▆▆▆▇▆▆▆▆▆▇▆▆▆▇▇▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▂▂▅██████▇▇▇▇▇▇▇▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/loss,▅▅▅▅▆██▇▇▅▂▂▁▁▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
best_val_acc,0.65605
best_val_acc_smoothed,0.67941
lr,0
train/acc,0.99907
train/loss,1.67801



================ s01 · seed 1 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1081개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.6994 acc 0.066 | val loss 2.7053 acc 0.000 avg 0.000 | lr 2.00e-04
[  2/80] train loss 2.6151 acc 0.141 | val loss 2.7028 acc 0.070 avg 0.035 | lr 2.00e-04
[  3/80] train loss 2.4975 acc 0.239 | val loss 2.7019 acc 0.070 avg 0.047 | lr 1.99e-04
[  4/80] train loss 2.3634 acc 0.403 | val loss 2.7042 acc 0.076 avg 0.072 | lr 1.99e-04
[  5/80] train loss 2.1983 acc 0.607 | val loss 2.7095 acc 0.076 avg 0.074 | lr 1.98e-04
[  6/80] train loss 2.0649 acc 0.766 | val loss 2.7215 acc 0.076 avg 0.076 | lr 1.97e-04
[  7/80] train loss 1.9409 acc 0.851 | val loss 2.7474 acc 0.076 avg 0.076 | lr 1.96e-04
[  8/80] train loss 1.8744 acc 0.896 | val loss 2.7821 acc 0.076 avg 0.076 | lr 1.95e-04
[  9/80] train loss 1.8334 acc 0.926 | val loss

...

lr,█████▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▄▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
train/acc,▁▂▅▆▇███████████████████████████████████
train/loss,█▆▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▂▂▃▄▅▆▆▇████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▂▃▄▅▆▆▇▇█████████████████████████
val/loss,▅▅▆▆██▇▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.64286
best_val_acc_smoothed,0.64502
lr,0
train/acc,1
train/loss,1.67509



================ s09 · seed 7 · 60프레임 ================


장치: cuda | 클래스: 15개
학습 1084개 · 검증 154개 클립 | 검증 화자 ['s09']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7062 acc 0.071 | val loss 2.7087 acc 0.058 avg 0.058 | lr 2.00e-04
[  2/80] train loss 2.5920 acc 0.152 | val loss 2.7080 acc 0.065 avg 0.062 | lr 2.00e-04
[  3/80] train loss 2.4778 acc 0.292 | val loss 2.7044 acc 0.065 avg 0.063 | lr 1.99e-04
[  4/80] train loss 2.3222 acc 0.456 | val loss 2.7036 acc 0.065 avg 0.065 | lr 1.99e-04
[  5/80] train loss 2.1429 acc 0.665 | val loss 2.7178 acc 0.065 avg 0.065 | lr 1.98e-04
[  6/80] train loss 1.9945 acc 0.812 | val loss 2.7512 acc 0.065 avg 0.065 | lr 1.97e-04
[  7/80] train loss 1.9036 acc 0.875 | val loss 2.8097 acc 0.065 avg 0.065 | lr 1.96e-04
[  8/80] train loss 1.8447 acc 0.913 | val loss 2.8670 acc 0.058 avg 0.063 | lr 1.95e-04
[  9/80] train loss 1.8036 acc 0.943 | val loss

lr,███████▇▇▇▇▇▇▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁
train/acc,▁▂▇█████████████████████████████████████
train/loss,█▇▅▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▂▃▄▅▆▇▇▇▇▇█████████████████████████
val/acc_smoothed,▁▁▁▁▁▁▁▁▂▂▅▅▆▆▆▇▇▇▇▇████████████████████
val/loss,▆▆▇▇███▆▄▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.6039
best_val_acc_smoothed,0.60606
lr,0
train/acc,1
train/loss,1.6747



화자         30프레임     60프레임        차이
----------------------------------------
s01        0.656     0.641    -0.015
s03        0.446     0.610    +0.164
s04        0.780     0.836    +0.056
s05        0.307     0.191    -0.116
s06        0.732     0.873    +0.141
s07        0.351     0.441    +0.090
s08        0.556     0.777    +0.221
s09        0.393     0.647    +0.254
----------------------------------------
평균         0.528     0.627    +0.099

화자 간 편차   30프레임 0.473 · 60프레임 0.682


## 9-1. 교차검증

### 에폭 80 → 120 (s01 · 3시드)

In [78]:
from src.ml.training.train import train

base80 = [0.713, 0.815, 0.694]
res = []
for sd in (42, 1, 7):
    print("")
    print("=== s01 · seed", sd, "· 120에폭 ===")
    a = train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
        val_speakers=["s01"],
        checkpoint_path=DRIVE_CHECKPOINTS / ("ep120_s01_seed" + str(sd) + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="ep120_s01_seed" + str(sd))["best"]
    res.append(round(a, 3))
    print("누적:", res)

m80 = sum(base80) / 3
m120 = sum(res) / 3
print("")
print("80에폭  ", base80, "평균", round(m80, 3), "폭", round(max(base80) - min(base80), 3))
print("120에폭 ", res, "평균", round(m120, 3), "폭", round(max(res) - min(res), 3))
print("차이", round(m120 - m80, 3))
# end


=== s01 · seed 42 · 120에폭 ===


장치: cuda | 클래스: 15개
학습 1081개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/120] train loss 2.7114 acc 0.078 | val loss 2.7044 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/120] train loss 2.6301 acc 0.117 | val loss 2.6984 acc 0.064 avg 0.067 | lr 2.00e-04
[  3/120] train loss 2.5205 acc 0.215 | val loss 2.6971 acc 0.064 avg 0.066 | lr 2.00e-04
[  4/120] train loss 2.3804 acc 0.388 | val loss 2.6992 acc 0.064 avg 0.064 | lr 1.99e-04
[  5/120] train loss 2.2514 acc 0.573 | val loss 2.7079 acc 0.064 avg 0.064 | lr 1.99e-04
[  6/120] train loss 2.0738 acc 0.736 | val loss 2.7275 acc 0.064 avg 0.064 | lr 1.99e-04
[  7/120] train loss 1.9455 acc 0.855 | val loss 2.7734 acc 0.076 avg 0.068 | lr 1.98e-04
[  8/120] train loss 1.8828 acc 0.892 | val loss 2.8378 acc 0.076 avg 0.072 | lr 1.98e-04
[  9/120] train loss 1.8363 acc 0.923 |

Exception in thread Thread-97 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_

KeyboardInterrupt: 

^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource_sharer.py", line 86, in get_connection


### 60프레임 · 120에폭 8화자 24런 (이어받기 지원)

In [ ]:
import json, time
from src.ml.training.train import train

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
SEEDS = [42, 1, 7]

LOG = DRIVE_ROOT / "cv60e120_results.json"
done = json.loads(LOG.read_text()) if LOG.exists() else []
seen = [tuple(x[:2]) for x in done]
print("이미 끝난 런", len(done), "/ 24")

t0 = time.time()
for sp in SPEAKERS:
    for sd in SEEDS:
        if (sp, sd) in seen:
            continue
        print("")
        print("========", sp, "· seed", sd, "· 60프레임 · 120에폭 ========")
        acc = train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=120, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("cv60e120_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="cv60e120_" + sp + "_seed" + str(sd))["best"]
        done.append([sp, sd, round(acc, 4)])
        LOG.write_text(json.dumps(done))
        print("저장 ·", len(done), "/ 24 · 경과", round((time.time() - t0) / 60), "분")
# end

## 9-2. 실험 러너 (권장)

9-1은 셀 안에서 반복문을 조립한다. 커널에 상태가 남아 2026-08-20에 모델 패치가
걸린 채로 실험 두 개가 오염된 적이 있다. `run_experiment.py`는 별도 프로세스로
돌고 설정을 결과에 함께 기록하므로 그 사고를 막는다. 새 실험은 이쪽을 쓴다.

- 결과는 `results/<이름>.json`에 쌓이고, 같은 이름으로 다시 돌리면 끝난 칸은 건너뛴다
- 설정이 이전 런과 다르면 멈춘다. 조건을 바꿀 때는 **이름을 새로 준다**
- `--summary`는 학습 없이 요약만 찍고, `--baseline <이름>`으로 기준선과 견준다
- 비교는 마지막 에폭 값으로 한다. 양쪽에 없으면 저장값으로 물러서며 그 사실을 찍는다
- `--deterministic`은 cuDNN 결정성을 켠다. 느려지므로 재현성을 잴 때만 쓴다

In [ ]:
# ═══ 실험 러너 · 화자 독립 교차검증 ═══
NAME  = "align"        # 최근 실험: 닮음 변환 정렬 크롭 · 192x96
SEEDS = "42 1 7"       # 한 칸 3시드가 최소 단위. 1시드로는 방향만 본다

cmd = (
    f"python scripts/run_experiment.py --name {NAME} --seeds {SEEDS}"
    f" --manifest {manifest_f60} --data-root {TRAIN_ROOT_F60}"
    f" --checkpoint-dir {DRIVE_CHECKPOINTS} --wandb-project lipreading"
)
print(cmd)
!{cmd}

In [ ]:
# 학습 없이 요약만 다시 본다. 기준선과 견주면 화자별 차이·짝 t검정·판정선까지 찍는다.
cmd = (
    f"python scripts/run_experiment.py --name {NAME} --summary"
    f" --baseline base60 --manifest {manifest_f60}"
)
print(cmd)
!{cmd}

## 10. 체크포인트 확인

In [ ]:
checkpoint = torch.load(DRIVE_CHECKPOINTS / "best.pt", map_location="cpu")

print(f"에폭 {checkpoint['epoch']}")
print(f"클래스 {checkpoint['num_classes']}개")
print(f"검증 정확도 {checkpoint['val_accuracy']:.3f}")
print(f"최근 평균 {checkpoint['smoothed_accuracy']:.3f}")
print(
    f"모델 hidden {checkpoint['hidden_dim']} · "
    f"layer {checkpoint['num_layer']} · dropout {checkpoint['dropout']}"
)

## 12. 증강 실험

2026-08-15에 공간 증강은 기각됐다(0.324 대 0.323).

### 증강 패치 원상복구

In [ ]:
import importlib, sys
from src.ml.preprocess.augmentation import pipeline

fresh = importlib.reload(pipeline)                    # 소스에서 원본을 새로 읽음
patched = sys.modules["src.ml.training.train"].VideoAugmentation
patched.__call__ = fresh.VideoAugmentation.__call__   # 학습 코드가 쥔 클래스에 되돌림
print("복구:", patched.__call__.__qualname__)          # VideoAugmentation.__call__ 이면 정상

### 시간축 증강 (s06 · 3시드)

In [ ]:
# ═══ 시간축 증강 실험 · 이 셀 하나만 실행 (재실행 안전) ═══
from pathlib import Path
import numpy as np
from src.ml.preprocess.augmentation.pipeline import VideoAugmentation
from src.ml.training.train import train

DRIVE_ROOT        = globals().get("DRIVE_ROOT", Path("/content/drive/MyDrive/hanium-lipreading"))
DRIVE_CHECKPOINTS = globals().get("DRIVE_CHECKPOINTS", DRIVE_ROOT / "checkpoints")
manifest_f60      = globals().get("manifest_f60", DRIVE_ROOT / "manifest_f60.csv")
TRAIN_ROOT_F60    = globals().get("TRAIN_ROOT_F60", Path("/content/data_f60"))
assert manifest_f60.exists(), f"매니페스트 없음: {manifest_f60}"

TIME_CROP_PROB = 0.5
TIME_CROP_MIN  = 0.75
SEEDS = [42, 1, 7]

# 원본을 클래스 속성에 한 번만 보관 → 몇 번 실행해도 진짜 원본이 유지된다
if not getattr(VideoAugmentation, "_taug_patched", False):
    VideoAugmentation._taug_orig = VideoAugmentation.__call__
    VideoAugmentation._taug_patched = True
ORIG = VideoAugmentation._taug_orig
assert ORIG.__qualname__ == "VideoAugmentation.__call__", f"원본이 아님: {ORIG.__qualname__}"

def call_with_time_aug(self, clip, return_details=False):
    if return_details:
        return ORIG(self, clip, True)
    frames = ORIG(self, clip, False)
    if self.rng.random() < TIME_CROP_PROB:
        T = len(frames)
        keep = int(T * self.rng.uniform(TIME_CROP_MIN, 1.0))
        if 2 <= keep < T:
            start = int(self.rng.integers(0, T - keep + 1))
            idx = np.linspace(start, start + keep - 1, T).round().astype(int)
            frames = frames[idx]
    return frames

VideoAugmentation.__call__ = call_with_time_aug
print(f"시간축 증강 적용 · 확률 {TIME_CROP_PROB} · 크롭 하한 {TIME_CROP_MIN}")
print(f"데이터 {TRAIN_ROOT_F60}\n")

results = {}
try:
    for seed in SEEDS:
        print(f"\n{'='*16} s06 · seed {seed} · 시간축 증강 {'='*16}")
        results[seed] = train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4,
            seed=seed, val_speakers=["s06"],
            checkpoint_path=DRIVE_CHECKPOINTS / f"taug_s06_seed{seed}.pt",
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name=f"taug_s06_seed{seed}",
        )["best"]
finally:
    VideoAugmentation.__call__ = ORIG
    print("\n[증강 원상복구 완료]")

v = [results[s] for s in SEEDS if s in results]
print(f"\n{'='*56}")
if len(v) == len(SEEDS):
    print(f"시간축 증강   {[f'{x:.3f}' for x in v]}   평균 {sum(v)/3:.3f} · 폭 {max(v)-min(v):.3f}")
else:
    print(f"완료 {len(v)}/{len(SEEDS)}시드   {[f'{x:.3f}' for x in v]}")
# 아래는 30프레임 시절 기준선이다. 60프레임 · 192x96에서는 다시 재야 한다.
print(f"기준선(30프레임 시절)  ['0.732', '0.783', '0.745']  평균 0.754")
print(f"지금 규격의 기준선은 없다. align 3시드가 나오면 그것과 견줄 것.")

### 재현성 확인 — 같은 시드 재실행

In [ ]:
from src.ml.training.train import train
from src.ml.preprocess.augmentation import VideoAugmentation

# 같은 시드를 다시 돌려 값이 얼마나 흔들리는지 잰다.
# s06 seed42가 0.809 / 0.739 / 0.822로 갈렸다. 폭 0.083이다.
#
# 한때 "시간축 증강 패치 오염"으로 봤으나 2026-08-20 2차 정정에서 철회했다.
# 원인은 오염이 아니라 재현 불가 자체이며, 남은 후보가 cuDNN 비결정성이다.
# True로 두 번 돌려 폭이 좁아지는지 본다. 켜면 느려진다.
DETERMINISTIC = False

print("call:", VideoAugmentation.__call__.__qualname__)
assert VideoAugmentation.__call__.__qualname__ == "VideoAugmentation.__call__", "패치가 걸려 있다"

tag = "repro_s06_seed42" + ("_det" if DETERMINISTIC else "")
acc = train(
    manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
    epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
    val_speakers=["s06"],
    checkpoint_path=DRIVE_CHECKPOINTS / (tag + ".pt"),
    num_workers=8, amp=True, ema_decay=0.998,
    hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
    deterministic=DETERMINISTIC,
    wandb_project="lipreading", run_name=tag)["best"]

print("")
print("재현 결과 ", round(acc, 3), "· cuDNN 결정성", "켬" if DETERMINISTIC else "끔")
print("  기존 관측  0.809 / 0.739 / 0.822  (폭 0.083)")
print("")
print("  한 번 돌려서는 아무 판정도 못 한다. 같은 설정으로 두 번 이상 돌린 값의")
print("  폭을 비교할 것. 결정성을 켠 쪽 폭이 눈에 띄게 좁으면 cuDNN이 원인이다.")

## 13. 랜드마크 실험

2화자 3시드에서 −0.023. 융합은 효과가 없었고 랜드마크 단독은 0.397이었다.

### 보조 특징(aux_f60.npz) 추출과 모델 패치

In [ ]:
import sys, urllib.request, unicodedata, time
import numpy as np, cv2, torch
from pathlib import Path
from torch import nn

from src.ml.preprocess.lip_crop import create_landmarker, LIP_LANDMARKS, lip_openness
from src.ml.preprocess.normalize import trim_to_speech, resample_frames
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.models.backbone import LipReadingBackbone
from src.ml.models.temporal import TemporalBiGRU
from src.ml.models.classification_head import ClassificationHead
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

AUX_DIM, PROJ_DIM, FRAMES = 44, 128, 60
OUT  = DRIVE_ROOT / "aux_f60.npz"
PART = DRIVE_ROOT / "aux_f60_partial.npz"
EXT  = [".mp4", ".avi", ".mov"]

if not OUT.exists():
    MODEL = Path("/content/hanium-lipreading/models/face_landmarker.task")
    if not MODEL.exists():
        MODEL.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task", str(MODEL))
        print("랜드마커 모델 내려받음")

    cand = []
    for p in sorted(DRIVE_ROOT.iterdir()):
        if not p.is_dir(): continue
        for d in [p] + [q for q in sorted(p.iterdir()) if q.is_dir()]:
            v = [q for q in d.iterdir() if q.suffix.lower() in EXT]
            if v: cand.append([len(v), d])
    cand.sort(reverse=True, key=lambda x: x[0])
    assert cand, "원본 영상을 Drive에서 못 찾았다"
    RAW_DIR = cand[0][1]
    print("RAW_DIR", RAW_DIR, cand[0][0], "개")

    vids = dict()
    for p in sorted(RAW_DIR.iterdir()):
        if p.suffix.lower() in EXT:
            vids[unicodedata.normalize("NFC", p.stem)] = p
    stems = [unicodedata.normalize("NFC", q.stem) for q in sorted(PROCESSED_F60.glob("*.npy"))]
    hit = [s for s in stems if s in vids]
    print("npy", len(stems), "개 · 원본 매칭", len(hit), "개")
    assert len(hit) > len(stems) * 0.95, "RAW_DIR 매칭 실패"

    names, feats = [], []
    if PART.exists():
        _p = np.load(PART, allow_pickle=False)
        names, feats = list(_p["names"]), list(_p["feats"])
        print("이어받기", len(names), "개")
    done = set(names)
    lmk = create_landmarker()
    t0 = time.time()
    try:
        for k, stem in enumerate(hit):
            if stem in done: continue
            cap = cv2.VideoCapture(str(vids[stem]))
            pts, ops = [], []
            while True:
                ok, fr = cap.read()
                if not ok: break
                h, w = fr.shape[:2]
                import mediapipe as mp
                res = lmk.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)))
                if not res.face_landmarks: continue
                L = res.face_landmarks[0]
                pts.append([[L[i].x * w, L[i].y * h] for i in LIP_LANDMARKS])
                ops.append(lip_openness(L, w, h))
            cap.release()
            if len(pts) < 2: continue
            idx = resample_frames(trim_to_speech(list(range(len(pts))), ops), FRAMES)
            P = np.array(pts, dtype=np.float32)[idx]
            O = np.array(ops, dtype=np.float32)[idx]
            P = P - P.mean(axis=1, keepdims=True)
            scale = float(np.median(np.linalg.norm(P[:, 0] - P[:, 10], axis=1))) + 1e-6
            P = P / scale
            dur = np.full((FRAMES, 1), len(pts) / float(FRAMES), dtype=np.float32)
            feats.append(np.concatenate([P.reshape(FRAMES, 42), O.reshape(FRAMES, 1), dur], axis=1).astype(np.float32))
            names.append(stem)
            if len(names) % 100 == 0:
                np.savez(PART, names=np.array(names), feats=np.array(feats))
                print(len(names), "/", len(hit), "·", round(time.time() - t0), "초")
    finally:
        lmk.close()
    np.savez(OUT, names=np.array(names), feats=np.array(feats, dtype=np.float32))
    print("저장", OUT, len(names), "개 ·", round(time.time() - t0), "초")

_z = np.load(OUT, allow_pickle=False)
AUX_FEATS = _z["feats"].astype(np.float32)
AUX_INDEX = dict(zip(list(_z["names"]), range(len(_z["names"]))))
print("보조 특징", AUX_FEATS.shape)

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(LipReadingModel, "_orig_init"):
    LipReadingModel._orig_init = LipReadingModel.__init__
    LipReadingModel._orig_forward = LipReadingModel.forward
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch

MISS = []

def getitem_aux(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    name = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    j = AUX_INDEX.get(name, -1)
    if j < 0:
        MISS.append(name)
        a = np.zeros((frames.shape[1], AUX_DIM), dtype=np.float32)
    else:
        a = AUX_FEATS[j]
    return frames, torch.from_numpy(a), label

def init_aux(self, num_classes, hidden_dim=256, num_layer=2, dropout=0.2,
             pretrained=False, freeze_backbone=False):
    nn.Module.__init__(self)
    self.backbone = LipReadingBackbone(pretrained=pretrained)
    if freeze_backbone:
        self.backbone.freeze_resnet()
    self.aux_proj = nn.Sequential(nn.Linear(AUX_DIM, PROJ_DIM), nn.ReLU())
    self.temporal = TemporalBiGRU(
        input_dim=self.backbone.feature_dim + PROJ_DIM,
        hidden_dim=hidden_dim, num_layer=num_layer, dropout=dropout)
    self.head = ClassificationHead(
        input_dim=self.temporal.output_dim, num_classes=num_classes, dropout=dropout)

def forward_aux(self, frames, aux):
    f = self.backbone(frames)
    f = torch.cat([f, self.aux_proj(aux)], dim=2)
    return self.head(self.temporal(f))

def run_epoch_aux(model, loader, criterion, device, optimizer=None,
                  amp=False, grad_clip=None, averager=None):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.set_grad_enabled(is_training):
        for frames, aux, labels in loader:
            frames = frames.to(device, non_blocking=True)
            aux = aux.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames, aux)
                loss = criterion(logits, labels)
            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None:
                    averager.update(model)
            total_loss += loss.float().item() * labels.size(0)
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_count += labels.size(0)
    return total_loss / total_count, total_correct / total_count

LipReadingDataset.__getitem__ = getitem_aux
LipReadingModel.__init__ = init_aux
LipReadingModel.forward = forward_aux
T.run_epoch = run_epoch_aux
print("패치 적용 완료")
# end

### 랜드마크 추가 학습 (s05 · s06 · seed 42)

In [ ]:
res = []
for sp in ["s05", "s06"]:
    print("")
    print("======== 랜드마크 추가 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("lm_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="lm_" + sp)["best"]
    res.append([sp, round(acc, 4)])
    print("누적:", res)
print("")
print("좌표 못 찾은 클립:", len(set(MISS)))
# end


### 랜드마크 추가 학습 (시드 1 · 7)

In [ ]:
res2 = []
for sp in ["s05", "s06"]:
    for sd in [1, 7]:
        print("")
        print("======== 랜드마크 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("lm_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="lm_" + sp + "_seed" + str(sd))["best"]
        res2.append([sp, sd, round(acc, 4)])
        print("누적:", res2)
# end

### 랜드마크 단독 분류 (영상 없이)

In [ ]:
import csv, unicodedata, numpy as np, torch
from torch import nn
from pathlib import Path

Z = np.load(DRIVE_ROOT / "aux_f60.npz", allow_pickle=False)
FEAT = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["feats"].astype(np.float32)))
rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
X, Y, S = [], [], []
for r in rows:
    stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
    if stem not in FEAT: continue
    X.append(FEAT[stem]); Y.append(int(r["label_id"])); S.append(r["speaker_id"])
X = torch.tensor(np.stack(X)); Y = torch.tensor(Y); S = np.array(S)
NC = int(Y.max()) + 1
print("클립", len(X), "· 차원", tuple(X.shape[1:]), "· 클래스", NC)

class AuxOnly(nn.Module):
    def __init__(self, nclass, dim=128):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(44, dim), nn.ReLU())
        self.gru = nn.GRU(dim, dim, num_layers=2, batch_first=True, bidirectional=True, dropout=0.3)
        self.head = nn.Linear(dim * 2, nclass)
    def forward(self, x):
        h, _ = self.gru(self.proj(x))
        return self.head(h.mean(dim=1))

BASE = dict(zip(["s01","s03","s04","s05","s06","s07","s08","s09"],
                [0.713, 0.554, 0.760, 0.193, 0.739, 0.363, 0.715, 0.407]))
dev = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, BS, SEED = 80, 16, 42
print("")
print("화자   랜드마크단독(저장)  (마지막)   영상기준선   우연")
res = []
for sp in sorted(set(S)):
    tr, va = np.where(S != sp)[0], np.where(S == sp)[0]
    mu = X[tr].reshape(-1, 44).mean(0); sg = X[tr].reshape(-1, 44).std(0) + 1e-6
    xt, yt = ((X[tr] - mu) / sg).to(dev), Y[tr].to(dev)
    xv, yv = ((X[va] - mu) / sg).to(dev), Y[va].to(dev)
    torch.manual_seed(SEED); np.random.seed(SEED)
    m = AuxOnly(NC).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=2e-4, weight_decay=0.01)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    crit = nn.CrossEntropyLoss(label_smoothing=0.1)
    g = torch.Generator().manual_seed(SEED)
    hist, best = [], 0.0
    for ep in range(EPOCHS):
        m.train()
        for i in torch.randperm(len(xt), generator=g).split(BS):
            opt.zero_grad(set_to_none=True)
            loss = crit(m(xt[i]), yt[i]); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        sch.step()
        m.eval()
        with torch.no_grad():
            acc = (m(xv).argmax(1) == yv).float().mean().item()
        hist.append(acc)
        best = max(best, float(np.mean(hist[-3:])))
    res.append([sp, round(best, 4), round(hist[-1], 4)])
    print(" ", sp, "   ", round(best, 3), "        ", round(hist[-1], 3), "     ",
          BASE.get(sp, 0), "     0.067")
print("")
print("평균  저장", round(np.mean([r[1] for r in res]), 3),
      "· 마지막", round(np.mean([r[2] for r in res]), 3),
      "· 영상기준선", round(np.mean(list(BASE.values())), 3))
# end

## 14. 화자 제외 실험

s05를 학습·평가 양쪽에서 뺀다. 7화자 1시드 +0.075.

In [ ]:
from src.ml.training import train as T
# end
import csv
(DRIVE_ROOT / "no_s05_results.json").unlink(missing_ok=True)
# end
rows = list(csv.DictReader(open(manifest_f60, encoding="utf-8")))
keep = [r for r in rows if r["speaker_id"] != "s05"]
manifest_no5 = DRIVE_ROOT / "manifest_f60_no_s05_3.csv"
with open(manifest_no5, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["clip_path", "label_id", "label_text", "speaker_id", "take"])
    w.writeheader()
    for r in keep: w.writerow(r)
print("클립", len(rows), "→", len(keep), "· 문구", len(set(r["label_text"] for r in keep)), "· 화자", len(set(r["speaker_id"] for r in keep)))

LOG5 = DRIVE_ROOT / "no_s05_results.json"
import json
done = json.loads(LOG5.read_text()) if LOG5.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s06", "s07", "s08", "s09"]:
    for sd in [1, 7]:
        if (sp, sd) in seen: continue
        print("")
        print("======== s05 제외 ·", sp, "· seed", sd, "========")
        acc = T.train(
            manifest_path=manifest_no5, data_root=TRAIN_ROOT_F60,
            epochs=80, batch_size=16, learning_rate=2e-4, seed=sd,
            val_speakers=[sp],
            checkpoint_path=DRIVE_CHECKPOINTS / ("no5_" + sp + "_seed" + str(sd) + ".pt"),
            num_workers=8, amp=True, ema_decay=0.998,
            hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
            wandb_project="lipreading", run_name="no5_" + sp + "_seed" + str(sd))["best"]
        done.append([sp, sd, round(acc, 4)])
        LOG5.write_text(json.dumps(done))
        print("누적:", done)
# end

클립 1238 → 1088 · 문구 15 · 화자 7

======== s05 제외 · s01 · seed 1 ========


lr,██▇▇▆▅▄▂▁
train/acc,▁▁▂▄▅▆▇██
train/loss,█▇▆▅▄▃▂▁▁
val/acc,▅▁▁▁▁▁███
val/acc_smoothed,▅▃▂▁▁▁▃▆█
val/loss,▁▁▁▁▁▂▄▆█
lr,0.0002
train/acc,0.92322
train/loss,1.8363
val/acc,0.07643
val/acc_smoothed,0.07643


장치: cuda | 클래스: 15개
학습 931개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7071 acc 0.059 | val loss 2.7073 acc 0.070 avg 0.070 | lr 2.00e-04
[  2/80] train loss 2.6296 acc 0.128 | val loss 2.7060 acc 0.070 avg 0.070 | lr 2.00e-04
[  3/80] train loss 2.5197 acc 0.215 | val loss 2.7072 acc 0.070 avg 0.070 | lr 1.99e-04
[  4/80] train loss 2.3419 acc 0.431 | val loss 2.7124 acc 0.076 avg 0.072 | lr 1.99e-04
[  5/80] train loss 2.1545 acc 0.679 | val loss 2.7220 acc 0.076 avg 0.074 | lr 1.98e-04
[  6/80] train loss 2.0050 acc 0.815 | val loss 2.7387 acc 0.076 avg 0.076 | lr 1.97e-04
[  7/80] train loss 1.8990 acc 0.896 | val loss 2.7639 acc 0.076 avg 0.076 | lr 1.96e-04
[  8/80] train loss 1.8234 acc 0.948 | val loss 2.8028 acc 0.076 avg 0.076 | lr 1.95e-04
[  9/80] train loss 1.7848 acc 0.968 | val loss 

lr,██████▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▄▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁
train/acc,▁▂▄▆▇███████████████████████████████████
train/loss,█▇▄▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/acc,▁▁▁▁▁▁▁▁▁▂▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇██████████▇▇▇▇▇
val/acc_smoothed,▁▁▁▁▁▁▁▁▁▁▁▁▂▂▄▅▅▅▆▆▆▆▇▇▇████████▇▇▇▇▇▇▇
val/loss,▆▆▆▆▆▇▇███▆▆▅▅▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_val_acc,0.6879
best_val_acc_smoothed,0.68577
lr,0
train/acc,1
train/loss,1.67099


누적: [['s01', 1, 0.6879]]

======== s05 제외 · s01 · seed 7 ========


장치: cuda | 클래스: 15개
학습 931개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균
[  1/80] train loss 2.7142 acc 0.066 | val loss 2.7100 acc 0.064 avg 0.064 | lr 2.00e-04
[  2/80] train loss 2.6331 acc 0.133 | val loss 2.7089 acc 0.064 avg 0.064 | lr 2.00e-04
[  3/80] train loss 2.5295 acc 0.224 | val loss 2.7078 acc 0.064 avg 0.064 | lr 1.99e-04
[  4/80] train loss 2.3529 acc 0.412 | val loss 2.7094 acc 0.070 avg 0.066 | lr 1.99e-04
[  5/80] train loss 2.1307 acc 0.697 | val loss 2.7147 acc 0.070 avg 0.068 | lr 1.98e-04
[  6/80] train loss 1.9848 acc 0.834 | val loss 2.7261 acc 0.070 avg 0.070 | lr 1.97e-04
[  7/80] train loss 1.8791 acc 0.902 | val loss 2.7395 acc 0.070 avg 0.070 | lr 1.96e-04
[  8/80] train loss 1.8181 acc 0.951 | val loss 2.7623 acc 0.070 avg 0.070 | lr 1.95e-04


## 15. 오디오 증류 실험

7화자에서 −0.040. 소리의 헷갈림이 입 모양과 맞지 않았다.

In [ ]:
import csv, json, unicodedata, subprocess, numpy as np, torch
from pathlib import Path
from torch import nn
from src.ml.training.dataset import LipReadingDataset
from src.ml.training import train as T

TEACH = DRIVE_ROOT / "teacher_f60.npz"
RAW_DIR = DRIVE_ROOT / "raw"
WHISPER = "openai/whisper-small"
TAU, ALPHA = 2.0, 0.5

rows = list(csv.DictReader(open(DRIVE_ROOT / "manifest_f60.csv", encoding="utf-8")))
PH = [t for _, t in sorted(set((int(r["label_id"]), r["label_text"]) for r in rows))]
print("문구", len(PH), "개")

if not TEACH.exists():
    subprocess.run(["pip", "install", "--quiet", "transformers", "accelerate"], check=False)
    from transformers import WhisperProcessor, WhisperForConditionalGeneration
    from transformers.modeling_outputs import BaseModelOutput
    dev = "cuda"
    proc = WhisperProcessor.from_pretrained(WHISPER)
    proc.tokenizer.set_prefix_tokens(language="korean", task="transcribe")
    wm = WhisperForConditionalGeneration.from_pretrained(WHISPER).to(dev).eval()

    seqs = [proc.tokenizer(p).input_ids for p in PH]
    L = max(len(s) for s in seqs)
    pad = proc.tokenizer.pad_token_id if proc.tokenizer.pad_token_id is not None else proc.tokenizer.eos_token_id
    lab = torch.full((len(PH), L), pad, dtype=torch.long)
    msk = torch.zeros(len(PH), L - 1)
    for i, s in enumerate(seqs):
        lab[i, :len(s)] = torch.tensor(s)
        msk[i, 3:len(s) - 1] = 1.0
    lab, msk = lab.to(dev), msk.to(dev)
    din, tgt = lab[:, :-1], lab[:, 1:]

    names, probs, nofail = [], [], 0
    tmp = Path("/content/_a.wav")
    for k, r in enumerate(rows):
        stem = unicodedata.normalize("NFC", Path(r["clip_path"]).stem)
        src = None
        for e in [".mp4", ".avi", ".mov"]:
            c = RAW_DIR / (stem + e)
            if c.exists(): src = c; break
        if src is None: continue
        subprocess.run(["ffmpeg", "-y", "-loglevel", "quiet", "-i", str(src),
                        "-ac", "1", "-ar", "16000", str(tmp)], check=False)
        if not tmp.exists() or tmp.stat().st_size < 2000:
            nofail += 1; continue
        import soundfile as sf
        wav, sr = sf.read(str(tmp))
        f = proc(wav, sampling_rate=16000, return_tensors="pt").input_features.to(dev)
        with torch.no_grad():
            enc = wm.model.encoder(f).last_hidden_state.repeat(len(PH), 1, 1)
            out = wm(decoder_input_ids=din, encoder_outputs=BaseModelOutput(last_hidden_state=enc))
            lp = torch.log_softmax(out.logits.float(), dim=-1)
            tok = lp.gather(2, tgt.unsqueeze(2)).squeeze(2)
            sc = (tok * msk).sum(1) / msk.sum(1)
            p = torch.softmax(sc / TAU, dim=0).cpu().numpy()
        names.append(stem); probs.append(p)
        tmp.unlink(missing_ok=True)
        if len(names) % 200 == 0: print(len(names), "/", len(rows))
    np.savez(TEACH, names=np.array(names), probs=np.array(probs, dtype=np.float32))
    print("교사 저장", len(names), "개 · 오디오 실패", nofail)

Z = np.load(TEACH, allow_pickle=False)
TP = dict(zip([unicodedata.normalize("NFC", n) for n in Z["names"]], Z["probs"]))
lid = dict(zip([unicodedata.normalize("NFC", Path(r["clip_path"]).stem) for r in rows],
               [int(r["label_id"]) for r in rows]))
top1 = np.mean([1.0 * (int(np.argmax(TP[s])) == lid[s]) for s in TP])
ent = np.mean([-float((TP[s] * np.log(TP[s] + 1e-9)).sum()) for s in TP])
print("")
print("교사 정확도", round(float(top1), 3), "· 평균 엔트로피", round(float(ent), 3),
      "· 균일분포 엔트로피", round(float(np.log(len(PH))), 3))

if not hasattr(LipReadingDataset, "_orig_getitem"):
    LipReadingDataset._orig_getitem = LipReadingDataset.__getitem__
if not hasattr(T, "_orig_run_epoch"):
    T._orig_run_epoch = T.run_epoch
UNIF = np.full(len(PH), 1.0 / len(PH), dtype=np.float32)

def getitem_kd(self, index):
    frames, label = LipReadingDataset._orig_getitem(self, index)
    stem = unicodedata.normalize("NFC", Path(self.rows[index]["clip_path"]).stem)
    return frames, torch.from_numpy(TP.get(stem, UNIF).copy()), label

def run_epoch_kd(model, loader, criterion, device, optimizer=None,
                 amp=False, grad_clip=None, averager=None):
    training = optimizer is not None
    model.train(training)
    tl = tc = tn = 0
    with torch.set_grad_enabled(training):
        for frames, soft, labels in loader:
            frames = frames.to(device, non_blocking=True)
            soft = soft.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                logits = model(frames)
                ce = criterion(logits, labels)
                kd = -(soft * torch.log_softmax(logits.float(), dim=1)).sum(1).mean()
                loss = (1.0 - ALPHA) * ce + ALPHA * kd
            if training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                if grad_clip: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if averager is not None: averager.update(model)
            tl += loss.float().item() * labels.size(0)
            tc += (logits.argmax(1) == labels).sum().item()
            tn += labels.size(0)
    return tl / tn, tc / tn

LipReadingDataset.__getitem__ = getitem_kd
T.run_epoch = run_epoch_kd
print("증류 패치 적용 · alpha", ALPHA, "· tau", TAU)

LOGK = DRIVE_ROOT / "kd_results.json"
done = json.loads(LOGK.read_text()) if LOGK.exists() else []
seen = [tuple(x[:2]) for x in done]
for sp in ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]:
    if (sp, 42) in seen: continue
    print("")
    print("======== 증류 ·", sp, "· seed 42 ========")
    acc = T.train(
        manifest_path=manifest_f60, data_root=TRAIN_ROOT_F60,
        epochs=80, batch_size=16, learning_rate=2e-4, seed=42,
        val_speakers=[sp],
        checkpoint_path=DRIVE_CHECKPOINTS / ("kd_" + sp + ".pt"),
        num_workers=8, amp=True, ema_decay=0.998,
        hidden_dim=300, num_layer=2, dropout=0.3, smoothing=3,
        wandb_project="lipreading", run_name="kd_" + sp)["best"]
    done.append([sp, 42, round(acc, 4)])
    LOGK.write_text(json.dumps(done))
    print("누적:", done)
# end

## 레거시 (옛 규격)

아래는 **30프레임 · 112x80 축 정렬** 시절 셀이다. 지금 데이터(60프레임 ·
192x96 닮음 변환)와 맞지 않아 그대로 돌리면 학습이 첫 배치에서 죽거나
옛 폴더를 건드린다.

바로 아래 셀이 **"모두 실행"을 여기서 멈춘다.** 필요한 셀은 손으로 하나씩 돌린다.

드라이브의 `processed_f60`과 `processed/`는 지우지 않았다. 192 규격이 아직
미검증이라(align 2화자 1시드) 되돌아갈 길을 남긴다.

In [ ]:
# ═══ 여기부터 레거시 ═══
# "모두 실행"을 여기서 멈춘다. 아래 셀들은 옛 규격이라 그대로 돌면
# 드라이브의 manifest.csv를 덮어쓰거나 옛 데이터를 로컬로 복사한다.
raise RuntimeError("의도된 정지. 여기까지가 현재 파이프라인이다. "
                   "아래 레거시 셀은 필요할 때 손으로 하나씩 돌릴 것.")

### 기본 전처리 (30프레임 · processed/)

In [63]:
import sys

sys.path.insert(0, str(REPO_DIR))

from src.ml.preprocess.vid2npy import run_batch

run_batch(raw_dir=DRIVE_RAW, processed_dir=DRIVE_PROCESSED)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed/s01_가래가있어요_11.npy
이미 존재함, 건너뜀: /content

### (60 프레임)

In [64]:
FRAMES = 60
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

from src.ml.preprocess import vid2npy
from src.ml.preprocess.lip_crop import crop_lip_frames
from src.ml.preprocess.normalize import normalize_frames

def process_video_f60(video_path, landmarker):
    lips, opennesses = crop_lip_frames(video_path, landmarker)
    if not lips:
        return None
    return normalize_frames(lips, opennesses, fixed_frame_count=FRAMES)

vid2npy.process_video = process_video_f60
vid2npy.run_batch(raw_dir=DRIVE_RAW, processed_dir=PROCESSED_F60)

이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_01.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_02.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_03.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_04.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_05.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_06.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_07.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_08.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_09.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s01_가래가있어요_10.npy
이미 존재함, 건너뜀: /content/drive/MyDrive/hanium-lipreading/processed_f60/s0

## 8. 매니페스트 생성

In [65]:
from scripts.build_manifest import build

manifest_path = DRIVE_ROOT / "manifest.csv"
build(processed_dir=DRIVE_PROCESSED, manifest_path=manifest_path)

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest.csv
  클립 1238개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요


In [66]:
import csv
from collections import Counter

with open(manifest_path, encoding="utf-8") as manifest_file:
    rows = list(csv.DictReader(manifest_file))

print(f"클립 {len(rows)}개")
print(f"화자별: {dict(Counter(row['speaker_id'] for row in rows))}")
print(f"문구별: {dict(Counter(row['label_text'] for row in rows))}")

클립 1238개
화자별: {'s01': 157, 's03': 148, 's04': 150, 's05': 150, 's06': 157, 's07': 171, 's08': 151, 's09': 154}
문구별: {'가래가있어요': 80, '간호사불러주세요': 79, '더워요': 86, '도와주세요': 82, '물주세요': 88, '배고파요': 87, '보호자불러주세요': 82, '숨쉬기힘들어요': 82, '아파요': 79, '어지러워요': 80, '자세바꿔주세요': 84, '진통제주세요': 86, '추워요': 82, '토할거같아요': 80, '화장실가고싶어요': 81}


### (60 프레임 · 로컬 복사와 검증 포함)

In [67]:
from scripts.build_manifest import build
from pathlib import Path
import shutil, time, numpy as np

manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

LOCAL_F60 = Path("/content/data60")
target = LOCAL_F60 / "processed"
expected = len(list(PROCESSED_F60.glob("*.npy")))

if target.exists() and len(list(target.glob("*.npy"))) != expected:
    shutil.rmtree(target)
if not target.exists():
    t = time.time()
    shutil.copytree(PROCESSED_F60, target)
    print(f"복사 {time.time() - t:.0f}초")

# 아까 겪은 0바이트·누락 확인
bad = [p.name for p in target.glob("*.npy")
       if p.stat().st_size == 0 or (np.load(p, mmap_mode="r") is None)]
n = len(list(target.glob("*.npy")))
print(f"로컬 {n}/{expected}개 · 손상 {len(bad)}개")
assert n == expected and not bad, "복사가 불완전합니다"

TRAIN_ROOT_F60 = LOCAL_F60

매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1238개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
복사 16초
로컬 1238/1238개 · 손상 0개


## 8-1. 학습 데이터를 로컬 디스크로 복사

In [68]:
import shutil
import time

LOCAL_ROOT = Path("/content/data")
LOCAL_PROCESSED = LOCAL_ROOT / "processed"

started = time.time()
if not LOCAL_PROCESSED.exists():
    shutil.copytree(DRIVE_PROCESSED, LOCAL_PROCESSED)

# 매니페스트의 clip_path가 "processed/..." 라서 data_root만 바꾸면 그대로 맞는다.
TRAIN_ROOT = LOCAL_ROOT
local_count = len(list(LOCAL_PROCESSED.glob("*.npy")))
print(f"로컬 npy {local_count}개 · {time.time() - started:.0f}초")

로컬 npy 1238개 · 0초


### 복사 검증

In [69]:
from pathlib import Path
import shutil, numpy as np

local = Path(TRAIN_ROOT) / "processed"

drive_names = {p.name for p in DRIVE_PROCESSED.glob("*.npy")}
local_names = {p.name for p in local.glob("*.npy")}
missing = drive_names - local_names
broken  = {p.name for p in local.glob("*.npy") if p.stat().st_size == 0}
fix = missing | broken

print(f"누락 {len(missing)} · 0바이트 {len(broken)} · 복구 대상 {len(fix)}개")
for name in sorted(fix):
    src = DRIVE_PROCESSED / name
    shutil.copy2(src, local / name)
    print(f"  {name}  ({src.stat().st_size:,}바이트)")

# 검증
bad = []
for p in local.glob("*.npy"):
    try:
        if p.stat().st_size == 0:
            raise ValueError
        np.load(p, mmap_mode="r")
    except Exception:
        bad.append(p.name)

print(f"\n로컬 {len(list(local.glob('*.npy')))}개 / Drive {len(drive_names)}개 · 손상 {len(bad)}개")
assert len(bad) == 0 and len(local_names | fix) == len(drive_names), "아직 안 맞습니다"

누락 0 · 0바이트 0 · 복구 대상 0개

로컬 1238개 / Drive 1238개 · 손상 0개


### 8-2 셋업 · 구 사본 (processed_f60)

In [72]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, shutil, time
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/hanium-lipreading")
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
PROCESSED_F60 = DRIVE_ROOT / "processed_f60"

n_drive = len(list(PROCESSED_F60.glob("*.npy")))
print("Drive processed_f60:", n_drive, "개")
assert n_drive == 1238, "processed_f60이 1234개가 아니다 - 전처리 이력 확인 필요"

REPO_DIR = Path("/content/hanium-lipreading")
os.chdir("/content")
if (REPO_DIR / ".git").exists():
    !cd {REPO_DIR} && git fetch origin && git checkout develop && git pull
else:
    !git clone -b develop https://github.com/HumanRhoid/hanium-lipreading.git {REPO_DIR\}
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
!pip install --quiet wandb

from scripts.build_manifest import build
manifest_f60 = DRIVE_ROOT / "manifest_f60.csv"
build(processed_dir=PROCESSED_F60, manifest_path=manifest_f60)

TRAIN_ROOT_F60 = Path("/content/data_f60")
LOCAL_F60 = TRAIN_ROOT_F60 / "processed"
started = time.time()
have = len(list(LOCAL_F60.glob("*.npy"))) if LOCAL_F60.exists() else 0
if have != n_drive:
    shutil.rmtree(TRAIN_ROOT_F60, ignore_errors=True)
    shutil.copytree(PROCESSED_F60, LOCAL_F60)

n_local = len(list(LOCAL_F60.glob("*.npy")))
bad = [p.name for p in LOCAL_F60.glob("*.npy") if p.stat().st_size == 0]
print("로컬", n_local, "개 ·", round(time.time() - started), "초 · 0바이트", len(bad), "개")
assert n_local == n_drive and not bad, "로컬 복사 불완전"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive processed_f60: 1238 개
Already on 'develop'
Your branch is up to date with 'origin/develop'.
Already up to date.
매니페스트 생성: /content/drive/MyDrive/hanium-lipreading/manifest_f60.csv
  클립 1238개 · 문구 15개 · 화자 8명
  라벨 매핑: 0=가래가있어요, 1=간호사불러주세요, 2=더워요, 3=도와주세요, 4=물주세요, 5=배고파요, 6=보호자불러주세요, 7=숨쉬기힘들어요, 8=아파요, 9=어지러워요, 10=자세바꿔주세요, 11=진통제주세요, 12=추워요, 13=토할거같아요, 14=화장실가고싶어요
로컬 1238 개 · 7 초 · 0바이트 0 개


### 9절 단일 학습 (baseline_s04 · 30프레임)

In [75]:
from src.ml.preprocess.augmentation import AugmentationConfig
from src.ml.training.train import train

# 증강 강도를 조절하려면 설정을 만들어 넘긴다. None이면 기본값을 쓴다.
strong_augmentation = AugmentationConfig(
    brightness_probability=0.7,
    contrast_probability=0.7,
    rotation_probability=0.6,
    shift_probability=0.6,
    zoom_probability=0.6,
)

best_accuracy = train(
    manifest_path=manifest_path,
    data_root=TRAIN_ROOT,
    epochs=80,
    batch_size=16,
    learning_rate=2e-4,
    seed=42,  # 가중치 초기값·데이터 순서·증강을 함께 고정한다
    val_speakers=["s04"],  # 설정 비교 시 고정. None이면 seed로 무작위 선택
    checkpoint_path=DRIVE_CHECKPOINTS / "best.pt",
    num_workers= 8,
    amp=True,
    hidden_dim=300,
    num_layer=2,
    dropout=0.3,
    weight_decay=0.01,
    smoothing=3,
    label_smoothing=0.1,  # 0이면 끔. 정답에 대한 과신을 줄인다
    grad_clip=1.0,  # 0이면 끔. 튀는 그래디언트를 잘라낸다
    ema_decay=0.998,  # 0이면 끔. 가중치 이동평균으로 검증한다
    augment=True,
    augmentation_config=None,  # strong_augmentation 으로 바꿔 강도 실험
    pretrained=False,  # True면 ImageNet 가중치 + ImageNet 입력 정규화
    freeze_backbone=False,  # pretrained와 함께 켜면 ResNet 층을 고정한다
    wandb_project="lipreading",
    run_name="baseline_s04",
)["best"]

KeyboardInterrupt: 

### 9-1 손수 교차검증 (run_experiment.py로 대체됨)

In [77]:
SEEDS = [42]  # [42, 1, 7] 로 늘리면 화자마다 여러 번 돌려 편차까지 본다

speakers = sorted({row["speaker_id"] for row in rows})
print(f"화자 {speakers} · 시드 {SEEDS}")

results = {}
for speaker in speakers:
    for seed in SEEDS:
        print(f"\n{'=' * 18} {speaker} · seed {seed} {'=' * 18}")
        results[(speaker, seed)] = train(
            manifest_path=manifest_path,
            data_root=TRAIN_ROOT,
            epochs=80,
            batch_size=16,
            learning_rate=2e-4,
            seed=seed,
            val_speakers=[speaker],
            checkpoint_path=DRIVE_CHECKPOINTS / f"cv_{speaker}_seed{seed}.pt",
            num_workers=8,
            amp=True,
            ema_decay=0.998,
            hidden_dim=300,
            num_layer=2,
            dropout=0.3,
            smoothing=3,
            wandb_project="lipreading",
            run_name=f"cv_{speaker}_seed{seed}",
        )["best"]

print(f"\n{'=' * 50}")
per_speaker = {}
for speaker in speakers:
    values = [results[(speaker, s)] for s in SEEDS]
    per_speaker[speaker] = sum(values) / len(values)
    detail = " ".join(f"{v:.3f}" for v in values)
    spread = f" (폭 {max(values) - min(values):.3f})" if len(values) > 1 else ""
    print(f"  {speaker}: {per_speaker[speaker]:.3f}   [{detail}]{spread}")

overall = sum(per_speaker.values()) / len(per_speaker)
gap = max(per_speaker.values()) - min(per_speaker.values())
print(f"\n전체 평균 {overall:.3f} · 화자 간 편차 {gap:.3f}")

화자 ['s01', 's03', 's04', 's05', 's06', 's07', 's08', 's09'] · 시드 [42]

================== s01 · seed 42 ==================


장치: cuda | 클래스: 15개
학습 1081개 · 검증 157개 클립 | 검증 화자 ['s01']
모델 hidden 300 · layer 2 · dropout 0.3 · wd 0.01 | 증강 켬 · 사전학습 끔 · 동결 끔
학습 파라미터 14.31M / 전체 14.31M | AMP 켬 · worker 8
label smoothing 0.1 · grad clip 1.0 · EMA 0.998 | 저장 기준 최근 3에폭 평균


RuntimeError: Caught RuntimeError in DataLoader worker process 3.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 57, in fetch
    return self.collate_fn(data)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 401, in default_collate
    return collate(batch, collate_fn_map=default_collate_fn_map)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 215, in collate
    collate(samples, collate_fn_map=collate_fn_map)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py", line 155, in collate
    r

## 11. 오답 분석

어느 문구끼리 헷갈리는지 본다.

### 예측 편향과 온도(τ) 보정

In [ ]:
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from src.ml.models.lip_reading_model import LipReadingModel
from src.ml.training.dataset import LipReadingDataset
from src.ml.training.train import split_by_speaker

SPEAKERS = ["s01", "s03", "s04", "s05", "s06", "s07", "s08", "s09"]
TAU_MAIN = 1.0          # 사전 선택값. 이걸로 판정한다

def probs_fold(speaker):
    ds = LipReadingDataset(manifest_path, TRAIN_ROOT)
    _, vi, _ = split_by_speaker(ds, val_speakers=[speaker])
    loader = DataLoader(Subset(ds, vi), batch_size=16, num_workers=2)
    ck = torch.load(DRIVE_CHECKPOINTS / f"cv_{speaker}_seed42.pt", map_location="cuda")
    m = LipReadingModel(num_classes=ck["num_classes"], hidden_dim=ck["hidden_dim"],
                        num_layer=ck["num_layer"], dropout=ck["dropout"]).cuda()
    m.load_state_dict(ck["model_state"]); m.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            with torch.autocast("cuda", dtype=torch.bfloat16):
                o = m(x.cuda())
            P.append(F.softmax(o.float(), 1).cpu()); Y.append(y)
    return torch.cat(P), torch.cat(Y), sorted({r["label_text"] for r in ds.rows})

fold = {}
for sp in SPEAKERS:
    fold[sp] = probs_fold(sp)
    print(f"{sp} 완료")

texts = fold["s01"][2]
C = len(texts)
total = sum(len(fold[s][1]) for s in SPEAKERS)

# ── 1. 예측 분포가 얼마나 치우쳤나 ──
pred_n, true_n = torch.zeros(C), torch.zeros(C)
for P, Y, _ in fold.values():
    pred_n += torch.bincount(P.argmax(1), minlength=C).float()
    true_n += torch.bincount(Y, minlength=C).float()

print(f"\n{'문구':<16}{'정답':>6}{'예측':>6}{'배율':>7}")
print("-" * 37)
for i in torch.argsort(pred_n / true_n, descending=True):
    print(f"{texts[i]:<16}{int(true_n[i]):>6}{int(pred_n[i]):>6}{pred_n[i] / true_n[i]:>7.2f}")

# ── 2. 보정 ──
def freq(speakers):
    n = torch.zeros(C)
    for s in speakers:
        n += torch.bincount(fold[s][0].argmax(1), minlength=C).float()
    return (n / n.sum()).clamp(min=1e-6)

base = sum((fold[s][0].argmax(1) == fold[s][1]).sum().item() for s in SPEAKERS) / total

print(f"\n{'τ':>5}{'상한(자기 화자)':>16}{'정직(타화자 추정)':>18}")
print("-" * 40)
for tau in (0.0, 0.25, 0.5, 0.75, 1.0):
    hi = ho = 0
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        others = [s for s in SPEAKERS if s != sp]
        hi += ((P / freq([sp]) ** tau).argmax(1) == Y).sum().item()
        ho += ((P / freq(others) ** tau).argmax(1) == Y).sum().item()
    mark = "  ← 판정" if tau == TAU_MAIN else ""
    print(f"{tau:>5.2f}{hi / total:>16.3f}{ho / total:>18.3f}{mark}")

print(f"\n보정 없음  {base:.3f}")

# ── 3. τ=1 정직 버전, 화자별 ──
print(f"\n{'화자':<6}{'보정전':>8}{'보정후':>8}{'차이':>8}")
print("-" * 32)
for sp in SPEAKERS:
    P, Y, _ = fold[sp]
    others = [s for s in SPEAKERS if s != sp]
    b = (P.argmax(1) == Y).float().mean().item()
    a = ((P / freq(others) ** TAU_MAIN).argmax(1) == Y).float().mean().item()
    print(f"{sp:<6}{b:>8.3f}{a:>8.3f}{a - b:>+8.3f}")

### 문구별 재현율과 주요 오답

In [ ]:
import collections

for target_name in ["도와주세요", "자세바꿔주세요", "숨쉬기힘들어요"]:
    t = texts.index(target_name)
    print(f"\n=== {target_name} ===")
    print(f"{'화자':<6}{'정답':>6}{'맞춤':>6}{'재현율':>8}  주요 오답")
    for sp in SPEAKERS:
        P, Y, _ = fold[sp]
        mask = Y == t
        n = int(mask.sum())
        if n == 0:
            continue
        pred = P[mask].argmax(1)
        hit = int((pred == t).sum())
        wrong = collections.Counter(texts[int(i)] for i in pred if int(i) != t)
        top = " · ".join(f"{a}×{c}" for a, c in wrong.most_common(2))
        print(f"{sp:<6}{n:>6}{hit:>6}{hit / n:>8.2f}  {top}")


### 문구별 움직임과 클립 중복률

In [ ]:
import numpy as np, csv, collections
from pathlib import Path

with open(manifest_path, encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

acc = collections.defaultdict(list)
for r in rows:
    a = np.load(Path(TRAIN_ROOT) / r["clip_path"])[:, :, :, 0].astype(np.float32)
    motion = np.abs(np.diff(a, axis=0)).mean()
    dup = np.mean([np.array_equal(a[i], a[i + 1]) for i in range(len(a) - 1)])
    acc[r["label_text"]].append((motion, dup))

print(f"{'문구':<16}{'움직임':>8}{'중복률':>8}{'클립':>6}")
print("-" * 40)
for m, ph, d, n in sorted((np.mean([x[0] for x in v]), ph,
                           np.mean([x[1] for x in v]), len(v))
                          for ph, v in acc.items()):
    print(f"{ph:<16}{m:>8.2f}{d:>8.2f}{n:>6}")